# Eventalign Pipeline — End-to-End Test

This notebook runs the full baleen eventalign pipeline on real test data:

1. **Input validation** — check files exist, BAMs are indexed
2. **BAM stats & contig filtering** — depth-based filtering
3. **f5c eventalign** — signal-level alignment
4. **Signal extraction & DTW** — per-position pairwise distance matrices
5. **Visualization** — heatmaps of DTW distance matrices

### Expected directory structure

```
testdata/
├── control_0/
│   ├── blow5/nanopore.blow5
│   └── fastq/pass.fq.gz
├── control_0.bam
├── control_0.bam.bai
├── control_0.bam.csi
├── native_0/
│   ├── blow5/nanopore.blow5
│   └── fastq/pass.fq.gz
├── native_0.bam
├── native_0.bam.bai
├── native_0.bam.csi
├── ref.fa
├── ref.fa.dict
└── ref.fa.fai
```

## 0. Setup & Path Configuration

In [ ]:
import logging
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Enable INFO-level logging so pipeline progress is visible
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
)

# Adjust this path to point to your testdata directory
TESTDATA = Path("../testdata").resolve()
assert TESTDATA.exists(), f"testdata directory not found: {TESTDATA}"

# ---------- File paths ----------
# Native sample
NATIVE_BAM   = TESTDATA / "native_0.bam"
NATIVE_FASTQ = TESTDATA / "native_0" / "fastq" / "pass.fq.gz"
NATIVE_BLOW5 = TESTDATA / "native_0" / "blow5" / "nanopore.blow5"

# Control (IVT) sample
IVT_BAM   = TESTDATA / "control_0.bam"
IVT_FASTQ = TESTDATA / "control_0" / "fastq" / "pass.fq.gz"
IVT_BLOW5 = TESTDATA / "control_0" / "blow5" / "nanopore.blow5"

# Reference
REF_FASTA = TESTDATA / "ref.fa"

# Verify all files exist
for label, path in [
    ("native BAM",   NATIVE_BAM),
    ("native FASTQ", NATIVE_FASTQ),
    ("native BLOW5", NATIVE_BLOW5),
    ("IVT BAM",      IVT_BAM),
    ("IVT FASTQ",    IVT_FASTQ),
    ("IVT BLOW5",    IVT_BLOW5),
    ("ref FASTA",    REF_FASTA),
]:
    status = "\u2705" if path.exists() else "\u274c MISSING"
    print(f"  {status}  {label:15s}  {path}")

print(f"\nAll paths verified from: {TESTDATA}")

## 1. BAM Statistics & Contig Filtering

Before running the full pipeline, inspect per-contig mapping statistics and
see which contigs pass depth filtering.

In [ ]:
from baleen.eventalign._bam import get_contig_stats, filter_contigs

native_stats = get_contig_stats(NATIVE_BAM)
ivt_stats = get_contig_stats(IVT_BAM)

print(f"Native BAM: {len(native_stats)} contigs with mapped reads")
print(f"IVT BAM:    {len(ivt_stats)} contigs with mapped reads")
print()

# Show top contigs by depth
print("--- Native (top 20 by mean depth) ---")
for s in sorted(native_stats.values(), key=lambda x: x.mean_depth, reverse=True)[:20]:
    print(f"  {s.contig:40s}  reads={s.mapped_reads:5d}  depth={s.mean_depth:.1f}")

print()
print("--- IVT (top 20 by mean depth) ---")
for s in sorted(ivt_stats.values(), key=lambda x: x.mean_depth, reverse=True)[:20]:
    print(f"  {s.contig:40s}  reads={s.mapped_reads:5d}  depth={s.mean_depth:.1f}")

In [ ]:
MIN_DEPTH = 15

passed_contigs, filter_results = filter_contigs(
    native_stats, ivt_stats, min_depth=MIN_DEPTH
)

print(f"Contigs passing filter (min_depth={MIN_DEPTH}): {len(passed_contigs)}")
for c in passed_contigs:
    nd = native_stats[c].mean_depth
    id_ = ivt_stats[c].mean_depth
    print(f"  {c:40s}  native_depth={nd:.1f}  ivt_depth={id_:.1f}")

print(f"\nContigs filtered out: {len(filter_results) - len(passed_contigs)}")
for fr in filter_results:
    if not fr.passed:
        print(f"  {fr.contig:40s}  reason={fr.reason.value}")

## 2. Run Full Pipeline

This calls `run_pipeline()` which orchestrates:
- f5c indexing
- BAM validation & contig filtering  
- Per-contig BAM splitting
- f5c eventalign
- Signal extraction & pairwise DTW computation

Results are saved to `output/` as a pickle file.

In [ ]:
from baleen.eventalign import run_pipeline, save_results, load_results

OUTPUT_DIR = Path("../output/eventalign_test").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results, metadata = run_pipeline(
    native_bam=NATIVE_BAM,
    native_fastq=NATIVE_FASTQ,
    native_blow5=NATIVE_BLOW5,
    ivt_bam=IVT_BAM,
    ivt_fastq=IVT_FASTQ,
    ivt_blow5=IVT_BLOW5,
    ref_fasta=REF_FASTA,
    min_depth=MIN_DEPTH,
    use_cuda=None,         # auto-detect GPU
    use_open_start=False,
    use_open_end=False,
    output_dir=OUTPUT_DIR,
    cleanup_temp=True,
    rna=True,
)

print(f"\n{'='*60}")
print(f"Pipeline complete!")
print(f"  f5c version:      {metadata.f5c_version}")
print(f"  Contigs total:    {metadata.n_contigs_total}")
print(f"  Contigs passed:   {metadata.n_contigs_passed_filter}")
print(f"  Contigs skipped:  {metadata.n_contigs_skipped}")
print(f"  use_cuda:         {metadata.use_cuda}")
print(f"  Results saved to: {OUTPUT_DIR / 'pipeline_results.pkl'}")

## 3. Inspect Results

Explore the pipeline output: per-contig summaries, number of positions with
DTW matrices, and matrix dimensions.

In [ ]:
# If you want to reload from disk instead of re-running:
# results, metadata = load_results(OUTPUT_DIR / "pipeline_results.pkl")

print(f"Contigs with results: {len(results)}\n")

for contig_name, contig_result in sorted(results.items()):
    n_pos = len(contig_result.positions)
    print(f"Contig: {contig_name}")
    print(f"  Native depth: {contig_result.native_depth:.1f}")
    print(f"  IVT depth:    {contig_result.ivt_depth:.1f}")
    print(f"  Positions with DTW matrices: {n_pos}")
    if n_pos > 0:
        # Show first few positions
        for pos, pr in list(contig_result.positions.items())[:5]:
            print(
                f"    pos={pos:6d}  kmer={pr.reference_kmer:6s}  "
                f"native_reads={pr.n_native_reads}  ivt_reads={pr.n_ivt_reads}  "
                f"matrix={pr.distance_matrix.shape}"
            )
        if n_pos > 5:
            print(f"    ... and {n_pos - 5} more positions")
    print()

## 4. Summary Statistics

In [ ]:
total_positions = 0
total_native_reads = 0
total_ivt_reads = 0
all_distances = []

for contig_result in results.values():
    for pr in contig_result.positions.values():
        total_positions += 1
        total_native_reads += pr.n_native_reads
        total_ivt_reads += pr.n_ivt_reads
        # Upper triangle (unique pairs)
        triu = pr.distance_matrix[np.triu_indices_from(pr.distance_matrix, k=1)]
        all_distances.extend(triu.tolist())

all_distances = np.array(all_distances)

print(f"Total positions with DTW results: {total_positions}")
print(f"Total native reads across positions: {total_native_reads}")
print(f"Total IVT reads across positions: {total_ivt_reads}")
print(f"Total unique pairwise distances: {len(all_distances)}")
if len(all_distances) > 0:
    print(f"\nDistance statistics:")
    print(f"  min:    {all_distances.min():.4f}")
    print(f"  max:    {all_distances.max():.4f}")
    print(f"  mean:   {all_distances.mean():.4f}")
    print(f"  median: {np.median(all_distances):.4f}")
    print(f"  std:    {all_distances.std():.4f}")

## 5. Visualize DTW Distance Matrices

For each contig, show heatmaps of the DTW distance matrices at selected positions.
The matrix rows/columns are ordered: **native reads first, then IVT reads**.
The dividing line between native and IVT is marked.

In [ ]:
def plot_dtw_matrix(pr, contig_name, ax=None):
    """Plot a single DTW distance matrix as a heatmap."""
    if ax is None:
        _, ax = plt.subplots(1, 1, figsize=(6, 5))

    mat = pr.distance_matrix
    n_nat = pr.n_native_reads
    n_ivt = pr.n_ivt_reads
    n_total = n_nat + n_ivt

    im = ax.imshow(mat, cmap="viridis", interpolation="nearest")
    plt.colorbar(im, ax=ax, shrink=0.8, label="DTW distance")

    # Draw boundary between native and IVT
    if n_nat > 0 and n_ivt > 0:
        ax.axhline(n_nat - 0.5, color="red", linewidth=1.5, linestyle="--")
        ax.axvline(n_nat - 0.5, color="red", linewidth=1.5, linestyle="--")

    ax.set_title(
        f"{contig_name}\npos={pr.position} kmer={pr.reference_kmer}\n"
        f"native={n_nat}, ivt={n_ivt}",
        fontsize=10,
    )
    ax.set_xlabel("Read index")
    ax.set_ylabel("Read index")

    # Label native vs IVT regions
    if n_total <= 30:
        labels = (
            [f"N{i}" for i in range(n_nat)]
            + [f"I{i}" for i in range(n_ivt)]
        )
        ax.set_xticks(range(n_total))
        ax.set_xticklabels(labels, fontsize=6, rotation=90)
        ax.set_yticks(range(n_total))
        ax.set_yticklabels(labels, fontsize=6)

    return ax

In [ ]:
MAX_POSITIONS_PER_CONTIG = 6  # how many positions to plot per contig

for contig_name, contig_result in sorted(results.items()):
    positions = contig_result.positions
    if not positions:
        print(f"Skipping {contig_name}: no positions with DTW results")
        continue

    # Pick evenly spaced positions to visualize
    sorted_positions = sorted(positions.keys())
    n_show = min(MAX_POSITIONS_PER_CONTIG, len(sorted_positions))
    if n_show <= MAX_POSITIONS_PER_CONTIG:
        selected = sorted_positions
    else:
        indices = np.linspace(0, len(sorted_positions) - 1, n_show, dtype=int)
        selected = [sorted_positions[i] for i in indices]

    n_cols = min(3, n_show)
    n_rows = (n_show + n_cols - 1) // n_cols
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(5 * n_cols, 4.5 * n_rows),
        squeeze=False,
    )
    fig.suptitle(f"Contig: {contig_name}", fontsize=13, fontweight="bold")

    for idx, pos in enumerate(selected):
        row, col = divmod(idx, n_cols)
        plot_dtw_matrix(positions[pos], contig_name, ax=axes[row][col])

    # Hide unused axes
    for idx in range(n_show, n_rows * n_cols):
        row, col = divmod(idx, n_cols)
        axes[row][col].set_visible(False)

    plt.tight_layout()
    plt.show()

## 6. Native vs IVT Distance Comparison

For each position, compare within-group distances (native-native, IVT-IVT)
against between-group distances (native-IVT). Modified positions should show
larger between-group distances.

In [ ]:
def decompose_distances(pr):
    """Split a distance matrix into within-native, within-IVT, and between-group distances."""
    mat = pr.distance_matrix
    n_nat = pr.n_native_reads
    n_ivt = pr.n_ivt_reads

    # Native-Native (upper triangle of top-left block)
    nn_block = mat[:n_nat, :n_nat]
    nn = nn_block[np.triu_indices_from(nn_block, k=1)]

    # IVT-IVT (upper triangle of bottom-right block)
    ii_block = mat[n_nat:, n_nat:]
    ii = ii_block[np.triu_indices_from(ii_block, k=1)]

    # Native-IVT (full off-diagonal block)
    ni = mat[:n_nat, n_nat:].ravel()

    return nn, ii, ni

In [ ]:
rows = []
for contig_name, contig_result in results.items():
    for pos, pr in contig_result.positions.items():
        nn, ii, ni = decompose_distances(pr)
        rows.append({
            "contig": contig_name,
            "position": pos,
            "kmer": pr.reference_kmer,
            "n_native": pr.n_native_reads,
            "n_ivt": pr.n_ivt_reads,
            "mean_native_native": float(nn.mean()) if len(nn) > 0 else np.nan,
            "mean_ivt_ivt": float(ii.mean()) if len(ii) > 0 else np.nan,
            "mean_native_ivt": float(ni.mean()) if len(ni) > 0 else np.nan,
        })

if rows:
    import pandas as pd
    df = pd.DataFrame(rows)
    df["ratio_between_within"] = df["mean_native_ivt"] / (
        (df["mean_native_native"] + df["mean_ivt_ivt"]) / 2
    )
    df = df.sort_values("ratio_between_within", ascending=False)
    print(f"Positions analyzed: {len(df)}")
    print()
    print(df.to_string(index=False, float_format="%.3f"))
else:
    print("No positions with results to analyze.")

In [ ]:
if rows:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: Distribution of mean distances by group
    ax = axes[0]
    ax.hist(df["mean_native_native"].dropna(), bins=30, alpha=0.6, label="Native-Native", color="tab:blue")
    ax.hist(df["mean_ivt_ivt"].dropna(), bins=30, alpha=0.6, label="IVT-IVT", color="tab:green")
    ax.hist(df["mean_native_ivt"].dropna(), bins=30, alpha=0.6, label="Native-IVT", color="tab:red")
    ax.set_xlabel("Mean DTW distance")
    ax.set_ylabel("Count (positions)")
    ax.set_title("Distribution of mean pairwise distances")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Right: Between/within ratio
    ax = axes[1]
    ratios = df["ratio_between_within"].dropna()
    ax.hist(ratios, bins=30, color="tab:purple", alpha=0.7)
    ax.axvline(1.0, color="gray", linestyle="--", linewidth=1.5, label="ratio=1.0")
    if len(ratios) > 0:
        ax.axvline(ratios.median(), color="red", linestyle="-", linewidth=1.5, label=f"median={ratios.median():.2f}")
    ax.set_xlabel("Between-group / Within-group distance ratio")
    ax.set_ylabel("Count (positions)")
    ax.set_title("Native-IVT separation")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 7. Per-Position Scatter: Within vs Between

Each point is a position. Positions above the diagonal have larger between-group
distance (potential modification sites).

In [ ]:
if rows:
    fig, ax = plt.subplots(figsize=(7, 7))

    within = (df["mean_native_native"] + df["mean_ivt_ivt"]) / 2
    between = df["mean_native_ivt"]

    ax.scatter(within, between, alpha=0.5, s=15, c="tab:blue")

    # Diagonal reference line
    lims = [
        min(within.min(), between.min()) * 0.95,
        max(within.max(), between.max()) * 1.05,
    ]
    ax.plot(lims, lims, "--", color="gray", linewidth=1, label="y=x")

    ax.set_xlabel("Mean within-group DTW distance")
    ax.set_ylabel("Mean between-group DTW distance (native vs IVT)")
    ax.set_title("Within vs Between-group distances per position")
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_aspect("equal")

    plt.tight_layout()
    plt.show()

---

**Done.** Results are saved at the output directory and can be reloaded with:

```python
from baleen.eventalign import load_results
results, metadata = load_results("output/eventalign_test/pipeline_results.pkl")
```